#### Searching the current directory and its parents for the project root

In [2]:
from pathlib import Path
import sys


def find_project_root(start: Path | None = None) -> Path:
    """
    Search the current directory and its parents for the project root.

    The project root must contain both:
    - config/__init__.py
    - src/__init__.py
    """
    start_path = (start or Path.cwd()).resolve()

    for candidate in (start_path, *start_path.parents):
        config_init = candidate / "config" / "__init__.py"
        src_init = candidate / "src" / "__init__.py"

        if config_init.is_file() and src_init.is_file():
            return candidate

    raise FileNotFoundError(
        "Project root could not be found. "
        "Confirm that config/__init__.py and src/__init__.py exist."
    )


PROJECT_ROOT = find_project_root()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project root: {PROJECT_ROOT}")
print(f"Project root added to sys.path: {str(PROJECT_ROOT) in sys.path}")

Project root: D:\Spain\Online Courses\10. Data Analysis Specialization\3. Python\Sprint 13\Assignment\eicu-lung-cancer-los
Project root added to sys.path: True


In [3]:
from __future__ import annotations

from pathlib import Path

import numpy as np
import pandas as pd

from config.paths import (
    COHORT_DIR,
    REDUCED_DATA_DIR,
)

#### Defining the files

In [4]:
LANDMARK_COHORT_FILE = (
    REDUCED_DATA_DIR
    / "lung_cancer_landmark24h_cohort_v002.csv"
)

APACHE_PREDVAR_FILE = (
    REDUCED_DATA_DIR
    / "apache_pred_var_model_features.csv"
)

APACHE_APSVAR_FILE = (
    REDUCED_DATA_DIR
    / "apache_aps_var_model_features.csv"
)

APACHE_RESULT_FILE = (
    REDUCED_DATA_DIR
    / "apache_patient_result_iva_lung_cancer.csv"
)

LAB_FEATURES_FILE = (
    REDUCED_DATA_DIR
    / "lab_first24h_features_v001.csv"
)

VITAL_FEATURES_FILE = (
    REDUCED_DATA_DIR
    / "vital_first24h_primary_features_v001.csv"
)

HISTORY_FEATURES_FILE = (
    REDUCED_DATA_DIR
    / "past_history_features_v001.csv"
)

HISTORY_MANIFEST_FILE = (
    COHORT_DIR
    / "qc"
    / "past_history_feature_manifest.csv"
)

#### Validating that every file exists

In [5]:
required_files = {
    "landmark cohort": LANDMARK_COHORT_FILE,
    "apachePredVar": APACHE_PREDVAR_FILE,
    "apacheApsVar": APACHE_APSVAR_FILE,
    "apachePatientResult": APACHE_RESULT_FILE,
    "laboratories": LAB_FEATURES_FILE,
    "vital signs": VITAL_FEATURES_FILE,
    "past history": HISTORY_FEATURES_FILE,
    "past-history manifest": HISTORY_MANIFEST_FILE,
}


missing_files = [
    f"{name}: {path}"
    for name, path in required_files.items()
    if not path.exists()
]

if missing_files:
    raise FileNotFoundError(
        "Required project files are missing:\n"
        + "\n".join(missing_files)
    )

print("All required feature files were found.")

All required feature files were found.


#### Using a safe loading function

In [6]:
def load_patient_level_table(
    path: Path,
    table_name: str,
) -> pd.DataFrame:
    """
    Load and validate a patient-level feature table.
    """

    dataframe = pd.read_csv(
        path,
        low_memory=False,
    )

    if "patientunitstayid" not in dataframe.columns:
        raise KeyError(
            f"{table_name} does not contain "
            "'patientunitstayid'."
        )

    dataframe["patientunitstayid"] = pd.to_numeric(
        dataframe["patientunitstayid"],
        errors="raise",
    ).astype("int64")

    duplicate_count = int(
        dataframe["patientunitstayid"]
        .duplicated()
        .sum()
    )

    if duplicate_count:
        raise ValueError(
            f"{table_name} contains "
            f"{duplicate_count:,} duplicate ICU-stay IDs."
        )

    print(
        f"{table_name}: "
        f"{len(dataframe):,} rows × "
        f"{dataframe.shape[1]:,} columns"
    )

    return dataframe

#### Loading everything

In [7]:
landmark_cohort = load_patient_level_table(
    LANDMARK_COHORT_FILE,
    "Landmark cohort",
)

apache_pred = load_patient_level_table(
    APACHE_PREDVAR_FILE,
    "apachePredVar",
)

apache_aps = load_patient_level_table(
    APACHE_APSVAR_FILE,
    "apacheApsVar",
)

apache_result = load_patient_level_table(
    APACHE_RESULT_FILE,
    "apachePatientResult",
)

lab_features = load_patient_level_table(
    LAB_FEATURES_FILE,
    "Laboratory features",
)

vital_features = load_patient_level_table(
    VITAL_FEATURES_FILE,
    "Vital-sign features",
)

history_features = load_patient_level_table(
    HISTORY_FEATURES_FILE,
    "Past-history features",
)

Landmark cohort: 1,389 rows × 37 columns
apachePredVar: 1,786 rows × 29 columns
apacheApsVar: 1,786 rows × 27 columns
apachePatientResult: 1,630 rows × 6 columns
Laboratory features: 1,389 rows × 112 columns
Vital-sign features: 1,389 rows × 39 columns
Past-history features: 1,389 rows × 40 columns


#### Validating the landmark cohort

In [8]:
assert len(landmark_cohort) == 1_389
assert landmark_cohort["patientunitstayid"].is_unique

if "prolonged_icu_los" not in landmark_cohort.columns:
    if "unitdischargeoffset" not in landmark_cohort.columns:
        raise KeyError(
            "Neither prolonged_icu_los nor "
            "unitdischargeoffset is available."
        )

    landmark_cohort["prolonged_icu_los"] = (
        pd.to_numeric(
            landmark_cohort["unitdischargeoffset"],
            errors="raise",
        )
        .gt(5 * 1440)
        .astype("int8")
    )


assert int(
    landmark_cohort["prolonged_icu_los"].sum()
) == 286

print("Landmark cohort validation passed.")

Landmark cohort validation passed.


#### Removing exact duplicates from apachePredVar

In [9]:
APACHE_PREDVAR_KEEP = [
    "patientunitstayid",
    "admitsource",
    "admitdiagnosis",
    "aids",
    "hepaticfailure",
    "lymphoma",
    "metastaticcancer",
    "leukemia",
    "immunosuppression",
    "cirrhosis",
    "diabetes",
    "myocardial_infarction_6m",
    "elective_surgery",
    "elective_surgery_missing",
    "active_treatment",
    "readmission",
    "vent_worst_rr_day1",
    "ventilated_day1",
    "intubated_day1",
    "noninvasive_ventilation_day1",
]

#### Validating that the selected fields exist

In [10]:
missing_apv_columns = set(
    APACHE_PREDVAR_KEEP
).difference(
    apache_pred.columns
)

if missing_apv_columns:
    raise KeyError(
        "Missing apachePredVar fields: "
        + ", ".join(sorted(missing_apv_columns))
    )

apache_pred = apache_pred[
    APACHE_PREDVAR_KEEP
].copy()

#### Prefixing the APACHE variables

In [11]:
def prefix_feature_columns(
    dataframe: pd.DataFrame,
    prefix: str,
) -> pd.DataFrame:
    dataframe = dataframe.copy()

    rename_map = {
        column: f"{prefix}{column}"
        for column in dataframe.columns
        if column != "patientunitstayid"
    }

    return dataframe.rename(
        columns=rename_map
    )

#### Applying prefixes

In [12]:
apache_pred = prefix_feature_columns(
    apache_pred,
    "apv_",
)

apache_aps = prefix_feature_columns(
    apache_aps,
    "aps_",
)

apache_result = prefix_feature_columns(
    apache_result,
    "apr_",
)

#### Merging all blocks

In [13]:
modeling_cohort = landmark_cohort.copy()

#### Using a validated merge helper

In [14]:
def merge_feature_block(
    base: pd.DataFrame,
    feature_block: pd.DataFrame,
    block_name: str,
) -> pd.DataFrame:
    """
    Left-join one patient-level feature block.
    """

    before_rows = len(base)

    merged = base.merge(
        feature_block,
        on="patientunitstayid",
        how="left",
        validate="one_to_one",
    )

    if len(merged) != before_rows:
        raise RuntimeError(
            f"{block_name} changed the cohort row count."
        )

    return merged

#### Merging in sequence

In [15]:
modeling_cohort = merge_feature_block(
    modeling_cohort,
    apache_pred,
    "apachePredVar",
)

modeling_cohort = merge_feature_block(
    modeling_cohort,
    apache_aps,
    "apacheApsVar",
)

modeling_cohort = merge_feature_block(
    modeling_cohort,
    apache_result,
    "apachePatientResult",
)

modeling_cohort = merge_feature_block(
    modeling_cohort,
    lab_features,
    "Laboratory features",
)

modeling_cohort = merge_feature_block(
    modeling_cohort,
    vital_features,
    "Vital-sign features",
)

modeling_cohort = merge_feature_block(
    modeling_cohort,
    history_features,
    "Past-history features",
)

#### Final structural checks

In [16]:
assert len(modeling_cohort) == 1_389

assert modeling_cohort[
    "patientunitstayid"
].is_unique

assert int(
    modeling_cohort[
        "prolonged_icu_los"
    ].sum()
) == 286

duplicate_column_names = (
    modeling_cohort.columns[
        modeling_cohort.columns.duplicated()
    ]
    .tolist()
)

if duplicate_column_names:
    raise ValueError(
        "Duplicate column names found: "
        + ", ".join(duplicate_column_names)
    )

print("All feature blocks merged successfully.")

All feature blocks merged successfully.


#### Creating block-availability flags

In [17]:
def any_feature_available(
    dataframe: pd.DataFrame,
    columns: list[str],
) -> pd.Series:
    return (
        dataframe[columns]
        .notna()
        .any(axis=1)
        .astype("int8")
    )

#### Identifying each feature block

In [18]:
apv_columns = [
    column
    for column in modeling_cohort.columns
    if column.startswith("apv_")
]

aps_columns = [
    column
    for column in modeling_cohort.columns
    if column.startswith("aps_")
]

apr_columns = [
    column
    for column in modeling_cohort.columns
    if column.startswith("apr_")
]

lab_columns = [
    column
    for column in modeling_cohort.columns
    if column.startswith("lab_")
]

vital_columns = [
    column
    for column in modeling_cohort.columns
    if column.startswith("vital_")
]

history_condition_columns = [
    column
    for column in modeling_cohort.columns
    if column.startswith("history_")
]

#### Creating flags

In [19]:
modeling_cohort["apv_available"] = (
    any_feature_available(
        modeling_cohort,
        apv_columns,
    )
)

modeling_cohort["aps_available"] = (
    any_feature_available(
        modeling_cohort,
        aps_columns,
    )
)

modeling_cohort["apr_available"] = (
    any_feature_available(
        modeling_cohort,
        apr_columns,
    )
)

#### Validating the other source flags

In [20]:
required_availability_flags = [
    "lab_any_available",
    "vital_any_available",
    "history_assessable_by_24h",
]

missing_flags = set(
    required_availability_flags
).difference(
    modeling_cohort.columns
)

if missing_flags:
    raise KeyError(
        "Missing availability flags: "
        + ", ".join(sorted(missing_flags))
    )

#### Defining variables that must not enter the model

In [21]:
TARGET_COLUMN = "prolonged_icu_los"

#### Identifier columns

In [22]:
identifier_columns = [
    column
    for column in modeling_cohort.columns
    if (
        column in {
            "patientunitstayid",
            "uniquepid",
            "patienthealthsystemstayid",
        }
        or column.lower().endswith("id")
    )
]

#### Outcome/leakage fields

In [23]:
LEAKAGE_KEYWORDS = [
    "unitdischarge",
    "hospitaldischarge",
    "actualicu",
    "actualhospital",
    "actualmortality",
    "death",
    "died",
    "mortalitystatus",
    "icu_los",
    "length_of_stay",
]

In [24]:
leakage_columns = [
    column
    for column in modeling_cohort.columns
    if (
        column != TARGET_COLUMN
        and any(
            keyword in column.lower()
            for keyword in LEAKAGE_KEYWORDS
        )
    )
]

#### APACHE predicted ICU LOS is benchmark-only

In [25]:
benchmark_only_columns = [
    column
    for column in modeling_cohort.columns
    if (
        "predictediculos" in column.lower()
        or "predicted_icu_los" in column.lower()
    )
]

Predicted mortality should also remain outside the first primary model

In [26]:
secondary_apache_prediction_columns = [
    column
    for column in modeling_cohort.columns
    if (
        "predictedicumortality" in column.lower()
        or "predictedhospitalmortality" in column.lower()
    )
]

#### Combining exclusions

In [27]:
model_exclusion_columns = sorted(
    set(
        identifier_columns
        + leakage_columns
        + benchmark_only_columns
        + secondary_apache_prediction_columns
        + [TARGET_COLUMN]
    )
)

#### Creating a feature manifest

In [28]:
def classify_feature(
    column: str,
) -> str:
    if column == TARGET_COLUMN:
        return "target"

    if column in identifier_columns:
        return "identifier"

    if column in leakage_columns:
        return "outcome_or_leakage"

    if column in benchmark_only_columns:
        return "benchmark_only"

    if column in secondary_apache_prediction_columns:
        return "secondary_apache_prediction"

    if column.endswith(
        (
            "_available",
            "_any_available",
            "_assessable_by_24h",
            "_missing",
            "_measurement_count",
            "_record_count",
        )
    ):
        return "availability_or_care_process"

    return "candidate_predictor"

In [29]:
feature_manifest = pd.DataFrame(
    {
        "feature": modeling_cohort.columns,
    }
)

feature_manifest["role"] = (
    feature_manifest["feature"]
    .map(classify_feature)
)

And the source block:

In [30]:
def identify_source_block(
    feature: str,
) -> str:
    if feature.startswith("apv_"):
        return "apachePredVar"

    if feature.startswith("aps_"):
        return "apacheApsVar"

    if feature.startswith("apr_"):
        return "apachePatientResult"

    if feature.startswith("lab_"):
        return "lab"

    if feature.startswith("vital_"):
        return "vital"

    if feature.startswith("history_"):
        return "pastHistory"

    return "cohort_or_demographic"

In [31]:
feature_manifest["source_block"] = (
    feature_manifest["feature"]
    .map(identify_source_block)
)

And missingness:

In [32]:
feature_manifest["missing_n"] = [
    int(modeling_cohort[column].isna().sum())
    for column in feature_manifest["feature"]
]

feature_manifest["missing_percent"] = (
    100
    * feature_manifest["missing_n"]
    / len(modeling_cohort)
)

feature_manifest["dtype"] = [
    str(modeling_cohort[column].dtype)
    for column in feature_manifest["feature"]
]

### Creating safe QC summaries
#### Columns by feature block

In [33]:
block_summary = (
    feature_manifest
    .groupby(
        [
            "source_block",
            "role",
        ],
        as_index=False,
    )
    .agg(
        number_of_columns=(
            "feature",
            "size",
        )
    )
)

#### Availability summary

In [34]:
availability_columns = [
    "apv_available",
    "aps_available",
    "apr_available",
    "lab_any_available",
    "vital_any_available",
    "history_assessable_by_24h",
]

In [35]:
availability_summary = pd.DataFrame(
    {
        "feature_block": availability_columns,
        "available_n": [
            int(modeling_cohort[column].sum())
            for column in availability_columns
        ],
    }
)

availability_summary["unavailable_n"] = (
    len(modeling_cohort)
    - availability_summary["available_n"]
)

availability_summary["coverage_percent"] = (
    100
    * availability_summary["available_n"]
    / len(modeling_cohort)
)

#### Target prevalence by source availability

In [36]:
availability_bias_tables = []


for availability_column in availability_columns:
    summary = (
        modeling_cohort
        .groupby(
            availability_column,
            dropna=False,
        )
        .agg(
            patient_count=(
                TARGET_COLUMN,
                "size",
            ),
            prolonged_count=(
                TARGET_COLUMN,
                "sum",
            ),
            prolonged_percent=(
                TARGET_COLUMN,
                lambda values: (
                    100 * values.mean()
                ),
            ),
        )
        .reset_index()
    )

    summary.insert(
        0,
        "availability_feature",
        availability_column,
    )

    summary = summary.rename(
        columns={
            availability_column:
                "availability_value"
        }
    )

    availability_bias_tables.append(
        summary
    )


availability_bias_summary = pd.concat(
    availability_bias_tables,
    ignore_index=True,
)

#### Exporting the frozen modelling cohort

In [37]:
MODELING_COHORT_CSV = (
    REDUCED_DATA_DIR
    / "modeling_cohort_v001.csv"
)

MODELING_COHORT_PARQUET = (
    REDUCED_DATA_DIR
    / "modeling_cohort_v001.parquet"
)


modeling_cohort.to_csv(
    MODELING_COHORT_CSV,
    index=False,
)

modeling_cohort.to_parquet(
    MODELING_COHORT_PARQUET,
    index=False,
    engine="pyarrow",
)

#### Saving the privacy-safe summaries

In [38]:
QC_DIR = COHORT_DIR / "qc"

QC_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


FEATURE_MANIFEST_FILE = (
    QC_DIR
    / "modeling_feature_manifest_v001.csv"
)

BLOCK_SUMMARY_FILE = (
    QC_DIR
    / "modeling_feature_block_summary_v001.csv"
)

AVAILABILITY_FILE = (
    QC_DIR
    / "modeling_block_availability_v001.csv"
)

AVAILABILITY_BIAS_FILE = (
    QC_DIR
    / "modeling_availability_target_summary_v001.csv"
)


feature_manifest.to_csv(
    FEATURE_MANIFEST_FILE,
    index=False,
)

block_summary.to_csv(
    BLOCK_SUMMARY_FILE,
    index=False,
)

availability_summary.to_csv(
    AVAILABILITY_FILE,
    index=False,
)

availability_bias_summary.to_csv(
    AVAILABILITY_BIAS_FILE,
    index=False,
)

#### Final validation report

In [39]:
candidate_predictor_count = int(
    feature_manifest[
        "role"
    ].eq("candidate_predictor").sum()
)

care_process_count = int(
    feature_manifest[
        "role"
    ].eq(
        "availability_or_care_process"
    ).sum()
)


print(
    "Rows in modeling_cohort:",
    f"{len(modeling_cohort):,}",
)

print(
    "Total columns:",
    f"{modeling_cohort.shape[1]:,}",
)

print(
    "Unique ICU stays:",
    f"{modeling_cohort['patientunitstayid'].nunique():,}",
)

print(
    "Duplicate ICU stays:",
    f"{modeling_cohort['patientunitstayid'].duplicated().sum():,}",
)

print(
    "Prolonged ICU stays:",
    (
        f"{int(modeling_cohort[TARGET_COLUMN].sum()):,} "
        f"({100 * modeling_cohort[TARGET_COLUMN].mean():.2f}%)"
    ),
)

print(
    "Candidate predictor columns:",
    f"{candidate_predictor_count:,}",
)

print(
    "Availability/care-process columns:",
    f"{care_process_count:,}",
)

print("\nFeature-block availability:")

display(
    availability_summary
)

Rows in modeling_cohort: 1,389
Total columns: 278
Unique ICU stays: 1,389
Duplicate ICU stays: 0
Prolonged ICU stays: 286 (20.59%)
Candidate predictor columns: 222
Availability/care-process columns: 34

Feature-block availability:


,feature_block,available_n,unavailable_n,coverage_percent
0,apv_available,1335,54,96.112311
1,aps_available,1335,54,96.112311
2,apr_available,1248,141,89.848812
3,lab_any_available,1349,40,97.120230
4,vital_any_available,1370,19,98.632109
5,history_assessable_by_24h,1360,29,97.912167
